# Compute noise robustness

**Do not run this notebook** — it takes a long time and is included for reproducibility only.
It shows how `robustness.pkl` (loaded by `fig5.ipynb`) was generated.

For each oscillatory matrix in all three ensembles, two robustness measures are computed:
- **IC robustness**: how large an initial-condition perturbation (η_ic) the orbit tolerates
  before the trajectory leaves the tube around the reference orbit.
- **Dynamics robustness**: same criterion but with OU noise injected during the trajectory.

Orbit membership is assessed via a KDTree tube criterion in the full 100D state space.
Results are saved to `robustness.pkl` as a list of dicts, one per oscillatory matrix.

In [ ]:
import os
os.environ['OMP_NUM_THREADS']      = '1'
os.environ['MKL_NUM_THREADS']      = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pickle
from scipy.spatial import cKDTree
from joblib import Parallel, delayed
from tqdm import tqdm
import sys
sys.path.append('../../..')
sys.path.append('../../../results')

from utils import (W_from_J, init_cond, dxdt,
                   solve_rk4_springBox, solve_rk4_noisy_springBox)

In [ ]:
# --- Load data ---
# Download from Zenodo [DOI] and place under data/fig2/
data_root = '../../../data/fig2'
sigmas    = ['sigma02', 'sigma05', 'sigma1']

data, data_normal = [], []
for s in sigmas:
    with open(f'{data_root}/{s}/Ws.pkl', 'rb') as f:
        data.append(pickle.load(f))
    with open(f'{data_root}/{s}/Ws_normal.pkl', 'rb') as f:
        data_normal.append(pickle.load(f))

solutions = [[d for d in ds if not np.all(d.get('W') == 0)] for ds in data]
matrices0 = [d['W'] for d in solutions[0]]
matrices1 = [d['W'] for d in solutions[1]]
matrices2 = [d['W'] for d in solutions[2]]

behaviours0 = np.load(f'{data_root}/behaviours/behaviours0.npy', allow_pickle=True).tolist()
behaviours1 = np.load(f'{data_root}/behaviours/behaviours1.npy', allow_pickle=True).tolist()
behaviours2 = np.load(f'{data_root}/behaviours/behaviours2.npy', allow_pickle=True).tolist()

In [ ]:
# --- Helper functions ---
OSC_LABELS = {'periodic', 'quasi-periodic'}

def find_first_oscillatory_index(behaviour_list, osc_labels=OSC_LABELS):
    for idx, lab in enumerate(behaviour_list):
        if lab in osc_labels:
            return idx
    return None

def build_panel1_inputs_from_ensembles(
        matrices0, matrices1, matrices2,
        behaviours0, behaviours1, behaviours2,
        a, b, c, ic_scale=0.1, verbose=True):
    """
    For each matrix that has at least one oscillatory IC, compute W and
    pick that IC as the starting point.

    Returns
    -------
    Ws, init_conds, ensemble_ids, meta
    """
    Ws, init_conds, ensemble_ids, meta = [], [], [], []
    for ens_id, (mats, behs) in enumerate(zip(
            [matrices0, matrices1, matrices2],
            [behaviours0, behaviours1, behaviours2])):
        kept = skipped = 0
        for i, (J, beh_list) in enumerate(zip(mats, behs)):
            idx = find_first_oscillatory_index(beh_list)
            if idx is None:
                skipped += 1
                continue
            W, _ = W_from_J(J, a, b, c)
            ics  = init_cond(J)
            x0   = ic_scale * ics[:, idx].copy()
            Ws.append(W)
            init_conds.append(x0)
            ensemble_ids.append(ens_id)
            meta.append(dict(ensemble_id=ens_id, matrix_index_in_ensemble=i,
                             ic_index=idx, behaviour_label=beh_list[idx]))
            kept += 1
        if verbose:
            print(f'Ensemble {ens_id}: kept={kept}, skipped={skipped}')
    return Ws, init_conds, ensemble_ids, meta

def estimate_current_rms_from_chunk(X_ref_chunk, W, external_input, tauE, tauI, a, b, c, bias):
    F = np.array([dxdt(x, W, external_input, tauE, tauI, a, b, c, bias)
                  for x in X_ref_chunk])
    d     = F.shape[1]
    quint = d // 5
    tau_vec = np.concatenate([float(tauE) * np.ones(4*quint),
                               float(tauI) * np.ones(d - 4*quint)])
    G     = F * tau_vec[None, :]
    g_rms = float(np.sqrt(np.mean(G**2)))
    G_rms = float(np.sqrt(np.mean(np.sum(G**2, axis=1))))
    return g_rms, G_rms

def ic_eta_to_std(eta_ic, d):
    return eta_ic / np.sqrt(d)

def eta_syn_to_sigma_ou(eta_syn, f_rms, tau_c):
    s_n = float(eta_syn) * float(f_rms)
    return np.sqrt(2.0) * s_n / np.sqrt(float(tau_c))

def compute_orbit_ellipsoid_and_scale(X_ref):
    T, d   = X_ref.shape
    center = X_ref.mean(axis=0)
    Xc     = X_ref - center
    cov    = np.cov(Xc, rowvar=False)
    eigvals, Q = np.linalg.eigh(cov)
    order  = np.argsort(eigvals)[::-1]
    eigvals, Q = eigvals[order], Q[:, order]
    r      = np.sqrt(np.maximum(eigvals, 0.0))
    Z      = Xc @ Q
    max_extent = np.max(np.abs(Z), axis=0)
    with np.errstate(divide='ignore', invalid='ignore'):
        scale_per_axis = np.where(r > 1e-12, max_extent / r, 1.0)
    scale_base = max(np.max(scale_per_axis), 1.0)
    a_axes = r * scale_base
    S      = float(np.sqrt(np.mean(a_axes**2)))
    return center, Q, a_axes, S

def same_orbit_from_precomputed_tree(tree, X_noisy_chunk, eps,
                                     frac_thresh=0.95, max_factor=2.0):
    dists, _    = tree.query(X_noisy_chunk)
    frac_inside = float(np.mean(dists <= eps))
    max_dist    = float(np.max(dists))
    same_orbit  = (frac_inside >= frac_thresh) and (max_dist <= max_factor * eps)
    return same_orbit, frac_inside, max_dist, dists

def critical_eta_from_partial(eta_grid, frac_same, p_thresh):
    eta_grid = np.asarray(eta_grid, float)
    frac_same = np.asarray(frac_same, float)
    mask = frac_same >= p_thresh
    return 0.0 if not np.any(mask) else float(np.max(eta_grid[mask]))

def should_stop_early(frac_same_history, p_thresh, consecutive=1):
    if len(frac_same_history) < consecutive:
        return False
    return all(p < p_thresh for p in frac_same_history[-consecutive:])

def run_trials_fraction_same(simulate_one_trial_fn, N_trials, p_thresh,
                             early_stop_trials=True, early_stop_success=True):
    successes = n_done = 0
    need_successes = int(np.ceil(p_thresh * N_trials))
    for k in range(N_trials):
        ok = bool(simulate_one_trial_fn())
        n_done += 1; successes += int(ok)
        if early_stop_trials:
            remaining = N_trials - n_done
            if successes + remaining < need_successes:
                break
            if early_stop_success and successes >= need_successes:
                break
    return successes / N_trials, successes, n_done

In [ ]:
# --- Per-matrix analysis ---
def analyze_matrix_panel1(
        W, x0, ensemble_id, tauE, tauI, T_total, dt,
        a, b, c, bias, t_start_idx, t_end_idx,
        eta_grid, alpha_full,
        frac_thresh=0.95, max_factor=2.0, N_trials=100, p_thresh=0.8,
        rng=None, early_stop_eta=True, early_stop_eta_consecutive=1,
        early_stop_trials=True, early_stop_success=True):

    if rng is None:
        rng = np.random.default_rng()
    d = W.shape[0]

    tvec, r_ref = solve_rk4_springBox(x0, W, tauE, tauI, T_total, dt, a, b, c, bias)
    X_ref_chunk = r_ref[t_start_idx:t_end_idx]

    _, _, _, S = compute_orbit_ellipsoid_and_scale(X_ref_chunk)
    eps  = alpha_full * S
    tree = cKDTree(X_ref_chunk)

    external_input = np.zeros(W.shape[0])
    g_rms, G_rms   = estimate_current_rms_from_chunk(
        X_ref_chunk, W=W, external_input=external_input,
        tauE=tauE, tauI=tauI, a=a, b=b, c=c, bias=bias)
    t_start_noise = float(tvec[t_start_idx])
    tau_c_ou = 0.01

    eta_ic_done, frac_same_ic   = [], []
    eta_dyn_done, frac_same_dyn = [], []

    # IC robustness
    for eta in eta_grid:
        sigma_ic = ic_eta_to_std(eta, d)
        def one_trial_ic():
            _, X_noisy = solve_rk4_noisy_springBox(
                x0, W, tauE, tauI, T_total, dt, a, b, c, bias,
                sigma_ic=sigma_ic, sigma_ou=0.0, tau_c=0.01,
                t_start_noise=None, rng=rng)
            same, _, _, _ = same_orbit_from_precomputed_tree(
                tree, X_noisy[t_start_idx:t_end_idx], eps,
                frac_thresh=frac_thresh, max_factor=max_factor)
            return same
        frac, _, _ = run_trials_fraction_same(one_trial_ic, N_trials, p_thresh,
                                              early_stop_trials, early_stop_success)
        eta_ic_done.append(float(eta)); frac_same_ic.append(float(frac))
        if early_stop_eta and should_stop_early(frac_same_ic, p_thresh, early_stop_eta_consecutive):
            break

    # Dynamics robustness
    for eta in eta_grid:
        sigma_ou = eta_syn_to_sigma_ou(eta, g_rms, tau_c_ou)
        def one_trial_dyn():
            _, X_noisy = solve_rk4_noisy_springBox(
                x0, W, tauE, tauI, T_total, dt, a, b, c, bias,
                sigma_ic=0.0, sigma_ou=sigma_ou, tau_c=tau_c_ou,
                t_start_noise=t_start_noise, rng=rng)
            same, _, _, _ = same_orbit_from_precomputed_tree(
                tree, X_noisy[t_start_idx:t_end_idx], eps,
                frac_thresh=frac_thresh, max_factor=max_factor)
            return same
        frac, _, _ = run_trials_fraction_same(one_trial_dyn, N_trials, p_thresh,
                                              early_stop_trials, early_stop_success)
        eta_dyn_done.append(float(eta)); frac_same_dyn.append(float(frac))
        if early_stop_eta and should_stop_early(frac_same_dyn, p_thresh, early_stop_eta_consecutive):
            break

    eta_ic_done  = np.asarray(eta_ic_done,  float)
    frac_same_ic = np.asarray(frac_same_ic, float)
    eta_dyn_done  = np.asarray(eta_dyn_done,  float)
    frac_same_dyn = np.asarray(frac_same_dyn, float)

    return {
        'ensemble_id'  : ensemble_id,
        'S'            : float(S),
        'eps'          : float(eps),
        'eta_grid_ic'  : eta_ic_done,
        'frac_same_ic' : frac_same_ic,
        'eta_grid_dyn' : eta_dyn_done,
        'frac_same_dyn': frac_same_dyn,
        'eta_ic_crit'  : critical_eta_from_partial(eta_ic_done,  frac_same_ic,  p_thresh),
        'eta_dyn_crit' : critical_eta_from_partial(eta_dyn_done, frac_same_dyn, p_thresh),
        'g_rms'        : float(g_rms),
        'G_rms'        : float(G_rms),
    }


def analyze_all_matrices_panel1_parallel(
        matrices, init_conds, ensemble_ids,
        tauE, tauI, T_total, dt, a, b, c, bias,
        t_start_idx, t_end_idx,
        eta_grid=None, alpha_2d=0.1,
        frac_thresh=0.95, max_factor=2.0,
        N_trials=100, p_thresh=0.8,
        n_jobs=None, base_seed=12345):

    if eta_grid is None:
        eta_grid = [0.01, 0.025, 0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 1.0]
    eta_grid = np.array(eta_grid, float)

    if n_jobs is None:
        n_jobs = max(1, os.cpu_count() // 2)

    ss = np.random.SeedSequence(base_seed)
    child_seeds = ss.spawn(len(matrices))

    def _worker(i, W_i, x0_i, ens_i, seed_i):
        res_i = analyze_matrix_panel1(
            W=W_i, x0=x0_i, ensemble_id=ens_i,
            tauE=tauE, tauI=tauI, T_total=T_total, dt=dt,
            a=a, b=b, c=c, bias=bias,
            t_start_idx=t_start_idx, t_end_idx=t_end_idx,
            eta_grid=eta_grid, alpha_full=alpha_2d,
            frac_thresh=frac_thresh, max_factor=max_factor,
            N_trials=N_trials, p_thresh=p_thresh,
            rng=np.random.default_rng(seed_i))
        res_i['matrix_index'] = i
        return res_i

    return Parallel(n_jobs=n_jobs, backend='loky', verbose=0)(
        delayed(_worker)(i, W_i, x0_i, ens_i, child_seeds[i])
        for i, (W_i, x0_i, ens_i) in enumerate(
            tqdm(list(zip(matrices, init_conds, ensemble_ids)),
                 desc='Matrices', total=len(matrices)))
    )

In [ ]:
# --- Simulation parameters ---
N    = 100
tauE = 0.1
tauI = 0.1
T_total = 15
dt   = 0.005
bias = np.zeros(N)
a    = np.ones(N)
c    = np.ones(N)
b    = np.zeros(N)
t_start_idx = 1000
t_end_idx   = 3000

Ws, init_conds, ensemble_ids, meta = build_panel1_inputs_from_ensembles(
    matrices0, matrices1, matrices2,
    behaviours0, behaviours1, behaviours2,
    a=a, b=b, c=c, ic_scale=0.1, verbose=True
)

In [ ]:
# --- Run parallel robustness analysis ---
# WARNING: takes a long time
results = analyze_all_matrices_panel1_parallel(
    matrices=Ws,
    init_conds=init_conds,
    ensemble_ids=ensemble_ids,
    tauE=tauE, tauI=tauI,
    T_total=T_total, dt=dt,
    a=a, b=b, c=c, bias=bias,
    t_start_idx=t_start_idx,
    t_end_idx=t_end_idx,
    eta_grid=[0.001, 0.0025, 0.005, 0.01, 0.02, 0.04, 0.08, 0.16, 0.32, 0.64, 1.0],
    alpha_2d=0.1,
    N_trials=100,
    p_thresh=0.8,
    n_jobs=7,
    base_seed=0,
)

In [ ]:
# --- Save ---
with open('../../../data/fig5/robustness.pkl', 'wb') as f:
    pickle.dump(results, f)
print(f'Saved {len(results)} records.')